In [98]:
import numpy as np
data = np.load('../tf_flowers_prepared.npz')

X_train = data['X_train']
y_train = data['y_train']

X_val = data['X_val']
y_val = data['y_val']

X_test = data['X_test']
y_test = data['y_test']

class_labels = data['class_names'].tolist()

## CNN Implementation

In [108]:
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.08),
    layers.RandomZoom(0.1),
])

In [114]:
import tensorflow as tf
from tensorflow.keras import layers, models, regularizers

cnn_model = models.Sequential([

    layers.Input(shape=(150, 150, 3)),

    # Data augmentation
    data_augmentation,

    # First convolutional block
    layers.Conv2D(
        32,
        (3, 3),
        activation='relu',
        kernel_regularizer=regularizers.l2(0.0001)
    ),
    layers.MaxPooling2D((2, 2)),

    # Second convolutional block
    layers.Conv2D(
        64,
        (3, 3),
        activation='relu',
        kernel_regularizer=regularizers.l2(0.0001)
    ),
    layers.MaxPooling2D((2, 2)),

    # Classification block
    layers.Flatten(),

    layers.Dense(
        64,
        activation='relu',
        kernel_regularizer=regularizers.l2(0.0001)
    ),

    layers.Dropout(0.5),

    # Output layer
    layers.Dense(3, activation='softmax')
])

cnn_model.summary()

Model: "sequential_17"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ sequential_15 (Sequential)      │ (None, 150, 150, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_28 (Conv2D)              │ (None, 148, 148, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_28 (MaxPooling2D) │ (None, 74, 74, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_29 (Conv2D)              │ (None, 72, 72, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_29 (MaxPooling2D) │ (None, 36, 36, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_14 (Flatten)            │ (None, 82944)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_27 (Dense)                │ (None, 64)             │     5,308,480 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_9 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_28 (Dense)                │ (None, 3)              │           195 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 5,328,067 (20.32 MB)

 Trainable params: 5,328,067 (20.32 MB)

 Non-trainable params: 0 (0.00 B)

In [115]:
cnn_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [116]:
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=8,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=3,
    min_lr=0.00001,
    verbose=1
)

In [118]:
history = cnn_model.fit(
    X_train,
    y_train,
    epochs=25,
    batch_size=32,
    validation_data=(X_val, y_val),
    callbacks=[early_stopping, reduce_lr]
)

Epoch 1/25
34/34 ━━━━━━━━━━━━━━━━━━━━ 13s 378ms/step - accuracy: 0.4954 - loss: 0.9848 - val_accuracy: 0.6056 - val_loss: 0.8403 - learning_rate: 0.0010
Epoch 2/25
34/34 ━━━━━━━━━━━━━━━━━━━━ 10s 299ms/step - accuracy: 0.5926 - loss: 0.9109 - val_accuracy: 0.6306 - val_loss: 0.8030 - learning_rate: 0.0010
Epoch 3/25
34/34 ━━━━━━━━━━━━━━━━━━━━ 10s 299ms/step - accuracy: 0.6222 - loss: 0.8740 - val_accuracy: 0.6583 - val_loss: 0.7838 - learning_rate: 0.0010
Epoch 4/25
34/34 ━━━━━━━━━━━━━━━━━━━━ 12s 347ms/step - accuracy: 0.6370 - loss: 0.8512 - val_accuracy: 0.6667 - val_loss: 0.7754 - learning_rate: 0.0010
Epoch 5/25
34/34 ━━━━━━━━━━━━━━━━━━━━ 10s 284ms/step - accuracy: 0.6398 - loss: 0.8078 - val_accuracy: 0.6694 - val_loss: 0.7760 - learning_rate: 0.0010
Epoch 6/25
34/34 ━━━━━━━━━━━━━━━━━━━━ 12s 351ms/step - accuracy: 0.6463 - loss: 0.8049 - val_accuracy: 0.6806 - val_loss: 0.7570 - learning_rate: 0.0010
Epoch 7/25
34/34 ━━━━━━━━━━━━━━━━━━━━ 9s 269ms/step - accuracy: 0.6593 - loss: 0.7

In [119]:
test_loss, test_accuracy = cnn_model.evaluate(X_test, y_test)

print("Test Accuracy:", test_accuracy)
print("Test Loss:", test_loss)

12/12 ━━━━━━━━━━━━━━━━━━━━ 1s 69ms/step - accuracy: 0.7528 - loss: 0.6734
Test Accuracy: 0.7527777552604675
Test Loss: 0.6733534336090088
